In [9]:
import cv2
import psycopg2
from ultralytics import YOLO
from datetime import datetime
import math
import numpy
conn = psycopg2.connect("dbname=user43 user=user43 password=m5q3x8tpc7vn host=2.nntc.nnov.ru port=5402")
video_path = 'Модуль А/fallback2.mp4'   # ПРОВЕРЬ НАЗВАНИЕ ВИДЕО!

In [11]:

# --- 1. КОНФИГУРАЦИЯ ---
VIDEO_PATH = 'Модуль А/fallback2.mp4'
DB_URL = "dbname=user43 user=user43 password=m5q3x8tpc7vn host=2.nntc.nnov.ru port=5402"
LINE_1_Y, LINE_2_Y = 350, 500
REAL_DIST_METERS = 5.0
OFFSET, DIST_THRESHOLD = 15, 70
TARGET_CLASSES = [2, 3, 5, 7]

# --- 2. ИНИЦИАЛИЗАЦИЯ ---
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
model = YOLO('yolo12s.pt')

conn = psycopg2.connect(DB_URL)
conn.autocommit = True
cursor = conn.cursor()

entry_frames, processed_ids = {}, set()
frame_count = 0

print(f"🚀 Запуск анализа. FPS: {fps:.2f} | Порог сближения: {DIST_THRESHOLD}px")

# --- 3. ГЛАВНЫЙ ЦИКЛ ОБРАБОТКИ ---
while cap.isOpened():
    success, frame = cap.read()
    if not success: break
    
    frame_count += 1
    now = datetime.now()

    results = model.track(frame, persist=True, verbose=False, tracker="botsort.yaml", classes=TARGET_CLASSES)
    annotated_frame = results[0].plot()

    cv2.line(annotated_frame, (0, LINE_1_Y), (frame.shape[1], LINE_1_Y), (255, 0, 0), 2)
    cv2.line(annotated_frame, (0, LINE_2_Y), (frame.shape[1], LINE_2_Y), (0, 0, 255), 2)

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.int().cpu().numpy()
        clss = results[0].boxes.cls.int().cpu().numpy()
        
        current_objs = []

        # Разбор объектов в кадре и приведение типов к Python float/int
        for i in range(len(ids)):
            x1, y1, x2, y2 = map(float, boxes[i])
            t_id, cls_name = int(ids[i]), model.names[int(clss[i])]
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
            
            current_objs.append({'id': t_id, 'cx': cx, 'cy': cy, 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2, 'cls': cls_name})

            # --- ЛОГИКА СКОРОСТИ ---
            if abs(cy - LINE_1_Y) < OFFSET and t_id not in entry_frames:
                entry_frames[t_id] = frame_count

            if abs(cy - LINE_2_Y) < OFFSET and t_id in entry_frames and t_id not in processed_ids:
                time_spent = (frame_count - entry_frames[t_id]) / fps
                if time_spent > 0:
                    speed = (REAL_DIST_METERS / time_spent) * 3.6
                    
                    if 1.0 < speed < 160:
                        cursor.execute("INSERT INTO vehicle_types (name_mode, first_seen, last_seen) VALUES (%s, %s, %s) ON CONFLICT (name_mode) DO UPDATE SET last_seen = EXCLUDED.last_seen RETURNING id;", (cls_name, now, now))
                        v_type_id = cursor.fetchone()[0]

                        cursor.execute("INSERT INTO traffic_raw (detection_time, vehicle_type_id, speed, track_id, x1, y1, x2, y2) VALUES (%s, %s, %s, %s, %s, %s, %s, %s);", (now, v_type_id, speed, t_id, x1, y1, x2, y2))
                        
                        processed_ids.add(t_id)
                        print(f"✅ Скорость {t_id}: {speed:.1f} км/ч")

        # --- ЛОГИКА ОПАСНЫХ СБЛИЖЕНИЙ ---
        for i in range(len(current_objs)):
            for j in range(i + 1, len(current_objs)):
                o1, o2 = current_objs[i], current_objs[j]
                
                # math.hypot - более быстрый и элегантный способ вычисления дистанции
                dist = math.hypot(o1['cx'] - o2['cx'], o1['cy'] - o2['cy'])
                
                if dist < DIST_THRESHOLD:
                    try:
                        cursor.execute("""
                            INSERT INTO dangerous_incidents (incident_time, car_a_id, car_b_id, distance_px, cx_a, cy_a, cx_b, cy_b)
                            VALUES (%s, %s, %s, %s, %s, %s, %s, %s) ON CONFLICT DO NOTHING
                        """, (now, o1['id'], o2['id'], dist, o1['cx'], o1['cy'], o2['cx'], o2['cy']))
                    except Exception as e:
                        print(f"⚠️ Ошибка БД: {e}")
                    
                    cv2.line(annotated_frame, (int(o1['cx']), int(o1['cy'])), (int(o2['cx']), int(o2['cy'])), (0, 165, 255), 2)
                    cv2.putText(annotated_frame, f"DANGER: {int(dist)}px", (int(o1['cx']), int(o1['cy'])-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    cv2.imshow('Traffic Monitoring System', cv2.resize(annotated_frame, (960, 540)))
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()
cursor.close()
conn.close()

KeyboardInterrupt: 